# Assignment 2B - RAG Pipeline

## Team - Group 20

| Name | ID | Contribution |
| :--- | :--- | :--- |
| ANMOL GUPTA | 2025ae05555 | 100% |
| AVISHEK KUMAR PRASAD | 2025ae05615 | 100% |
| PATIL SOHAM SHASHIKANT | 2025ae05221 | 100% |
| PRAFFUL NAMDEV | 2025ae05804 | 100% |

Run on Google Colab with a T4 GPU.

## What we reuse from Assignment 1

- **Documents:** the cleaned `.txt` files in `domain_corpus/` (45 documents: Intune / Autopilot docs, Ubuntu Server docs, ITSM and ServiceNow pages).
- **Generator:** our fine-tuned model `final_it_qlora_merged` (SmolLM2-360M, continued pre-training + QLoRA).
- **Queries:** our Assignment 1 eval set was 10 helpdesk instruction pairs (VPN error, locked account, full disk, ...). The expected answers there were written by us and are not in the corpus, so there is no answer span to mark for them. We checked this below. So we wrote 10 new questions in the same domain whose answers are in the corpus documents.

Before running, upload `domain_corpus.zip` and `final_it_qlora_merged.zip` (from the Assignment 1 run) to the Colab session; the next cells unzip them.

Note: Part A2 says 100 and 400 tokens but the table says 500. We followed the text and the checklist and used 100 and 400.

## Install packages

In [1]:
!pip install -q faiss-cpu rank_bm25 sentence-transformers
!pip install -q -U 'bitsandbytes>=0.46.1'

In [9]:
import os
print(os.path.getsize('/content/final_it_qlora_merged.zip'))

246921348


In [10]:
import os
import re
import time
import zipfile
import numpy as np
import pandas as pd
import torch

# Upload these two files to the Colab session first (left sidebar > Files > Upload):
#   domain_corpus.zip   and   final_it_qlora_merged.zip
def unzip_once(zip_path, target):
    assert os.path.exists(zip_path), zip_path + ' not found - upload it to /content first'
    if not os.path.isdir(target):
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(target if target.endswith('domain_corpus') else os.path.dirname(target))

unzip_once('/content/domain_corpus.zip', '/content/domain_corpus')
unzip_once('/content/final_it_qlora_merged.zip', '/content/final_it_qlora_merged')

CORPUS_DIR = '/content/domain_corpus'
A1_MODEL_PATH = '/content/final_it_qlora_merged'

# the zip contains the same Ubuntu Networking page twice (file name ends with ' (1).txt'); keep one copy
for f in os.listdir(CORPUS_DIR):
    if f.endswith(' (1).txt'):
        os.remove(os.path.join(CORPUS_DIR, f))
        print('removed duplicate:', f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')


removed duplicate: ubuntu_11_Networking - Ubuntu Server documentation (1).txt
Device: Tesla T4


In [11]:
docs = []
for fname in sorted(os.listdir(CORPUS_DIR)):
    if fname.endswith('.txt'):
        with open(os.path.join(CORPUS_DIR, fname), encoding='utf-8') as f:
            docs.append({'doc_id': fname, 'text': f.read()})

print('Documents loaded:', len(docs))
print('Total words:', sum(len(d['text'].split()) for d in docs))

Documents loaded: 43
Total words: 112206


# Part A - Dense search and chunk size

### Queries and answer spans

For each query we copied the sentence (or part of it) from the corpus that answers it. A retrieved chunk counts as a **hit** if it contains this span. When matching we ignore case and punctuation, so small formatting differences in the scraped text don't matter.

| ID | Query | Answer span | Document |
| --- | --- | --- | --- |
| Q01 | What are the two modes of operation of AppArmor profiles, and how do they differ? | profile violations are permitted and logged | Ubuntu - AppArmor |
| Q02 | How do you fully disable AppArmor on Ubuntu 24.04 LTS or later? | add apparmor=0 to GRUB_CMDLINE_LINUX | Ubuntu - AppArmor |
| Q03 | In OpenSSH algorithm lists such as Ciphers or MACs, what do the +, - and ^ prefixes do? | will be appended to the end of the default set | Ubuntu - OpenSSH crypto configuration |
| Q04 | How can I check the effective Ciphers setting of an OpenSSH server after editing its configuration? | use the -T parameter and grep the output for the configuration key | Ubuntu - OpenSSH crypto configuration |
| Q05 | What are the default PostgreSQL client authentication methods on Ubuntu for local and host connections? | peer is the default authentication method used for local connections | Ubuntu - PostgreSQL |
| Q06 | How do I allow other computers to connect to my PostgreSQL server on Ubuntu? | Locate the line: #listen_addresses = 'localhost' and change it to | Ubuntu - PostgreSQL |
| Q07 | What is the recommended way to persist data from Docker containers, and why is it better than the container layer? | The recommended way to persist data to and from Docker containers is by using volumes | Ubuntu - Docker storage, networking and logging |
| Q08 | Which logging driver does Docker use by default, and what does it do? | it is the default logging driver. It writes logs in a file in JSON format | Ubuntu - Docker storage, networking and logging |
| Q09 | Does Windows Autopilot need to re-image new devices? What does it do instead? | Instead of re-imaging the device, the existing Windows installation can be transformed into | Microsoft Learn - Windows Autopilot |
| Q10 | What subscription is needed to configure Windows Autopilot auto-enrollment into Microsoft Intune? | Requires a Microsoft Entra ID P1 or P2 subscription for configuration | Microsoft Learn - Windows Autopilot |

For D1.3 we also use one question that the corpus cannot answer: *How do I reset a forgotten BIOS supervisor password on a Lenovo ThinkPad?*

In [12]:
queries = [
    {'id': 'Q01',
     'query': 'What are the two modes of operation of AppArmor profiles, and how do they differ?',
     'answer': 'Complain (learning) mode, where profile violations are permitted and logged, which is useful for testing and developing new profiles, and enforce (confined) mode, which enforces the profile policy and also logs violations.',
     'span': 'profile violations are permitted and logged'},
    {'id': 'Q02',
     'query': 'How do you fully disable AppArmor on Ubuntu 24.04 LTS or later?',
     'answer': 'From 24.04 AppArmor is built into the kernel, so you add apparmor=0 to GRUB_CMDLINE_LINUX in /etc/default/grub, run sudo update-grub and reboot.',
     'span': 'add apparmor=0 to GRUB_CMDLINE_LINUX'},
    {'id': 'Q03',
     'query': 'In OpenSSH algorithm lists such as Ciphers or MACs, what do the +, - and ^ prefixes do?',
     'answer': '+ appends the algorithms to the end of the default set, - removes them from the default set and ^ puts them at the beginning of the default set. A list without a prefix replaces the default set.',
     'span': 'will be appended to the end of the default set'},
    {'id': 'Q04',
     'query': 'How can I check the effective Ciphers setting of an OpenSSH server after editing its configuration?',
     'answer': 'Run sshd with the -T parameter and grep for the setting, e.g. sudo sshd -T | grep ciphers. It shows the changed value without restarting the service (a restart of ssh.service is only needed to apply it).',
     'span': 'use the -T parameter and grep the output for the configuration key'},
    {'id': 'Q05',
     'query': 'What are the default PostgreSQL client authentication methods on Ubuntu for local and host connections?',
     'answer': 'peer for local connections and scram-sha-256 for host connections (host used md5 until Ubuntu 21.10).',
     'span': 'peer is the default authentication method used for local connections'},
    {'id': 'Q06',
     'query': 'How do I allow other computers to connect to my PostgreSQL server on Ubuntu?',
     'answer': 'Only local connections are allowed by default. Edit /etc/postgresql/*/main/postgresql.conf and change #listen_addresses = localhost to listen_addresses = * so it listens on all interfaces, allow the clients in pg_hba.conf and restart PostgreSQL.',
     'span': "Locate the line: #listen_addresses = 'localhost' and change it to"},
    {'id': 'Q07',
     'query': 'What is the recommended way to persist data from Docker containers, and why is it better than the container layer?',
     'answer': 'Volumes. They are managed by Docker, easier to back up and migrate than bind mounts, can be shared safely between containers, support volume drivers for remote or cloud storage, and do not increase the size of the container like data in the writable layer does.',
     'span': 'The recommended way to persist data to and from Docker containers is by using volumes'},
    {'id': 'Q08',
     'query': 'Which logging driver does Docker use by default, and what does it do?',
     'answer': 'json-file. It writes the container logs (what goes to STDOUT and STDERR) to a file in JSON format.',
     'span': 'it is the default logging driver. It writes logs in a file in JSON format'},
    {'id': 'Q09',
     'query': 'Does Windows Autopilot need to re-image new devices? What does it do instead?',
     'answer': 'No. It uses the OEM-optimized Windows already on the device and transforms it into a business-ready state: applies settings and policies, installs apps and can change the edition, e.g. Pro to Enterprise.',
     'span': 'Instead of re-imaging the device, the existing Windows installation can be transformed into'},
    {'id': 'Q10',
     'query': 'What subscription is needed to configure Windows Autopilot auto-enrollment into Microsoft Intune?',
     'answer': 'A Microsoft Entra ID P1 or P2 subscription.',
     'span': 'Requires a Microsoft Entra ID P1 or P2 subscription for configuration'},
]

unanswerable = {'id': 'U01', 'query': 'How do I reset a forgotten BIOS supervisor password on a Lenovo ThinkPad?'}

pd.DataFrame(queries)[['id', 'query', 'span']]

,id,query,span
0,Q01,What are the two modes of operation of AppArmo...,profile violations are permitted and logged
1,Q02,How do you fully disable AppArmor on Ubuntu 24...,add apparmor=0 to GRUB_CMDLINE_LINUX
2,Q03,In OpenSSH algorithm lists such as Ciphers or ...,will be appended to the end of the default set
3,Q04,How can I check the effective Ciphers setting ...,use the -T parameter and grep the output for t...
4,Q05,What are the default PostgreSQL client authent...,peer is the default authentication method used...
5,Q06,How do I allow other computers to connect to m...,Locate the line: #listen_addresses = 'localhos...
6,Q07,What is the recommended way to persist data fr...,The recommended way to persist data to and fro...
7,Q08,Which logging driver does Docker use by defaul...,it is the default logging driver. It writes lo...
8,Q09,Does Windows Autopilot need to re-image new de...,"Instead of re-imaging the device, the existing..."
9,Q10,What subscription is needed to configure Windo...,Requires a Microsoft Entra ID P1 or P2 subscri...


In [13]:
def normalize(text):
    # lowercase and replace punctuation with spaces, so 'apparmor=0' and 'apparmor = 0' both match
    text = re.sub(r'[^a-z0-9]+', ' ', text.lower())
    return ' ' + ' '.join(text.split()) + ' '

def contains_span(text, span):
    return normalize(span) in normalize(text)

# check that every span really is in the corpus
for q in queries:
    found_in = [d['doc_id'] for d in docs if contains_span(d['text'], q['span'])]
    print(q['id'], '->', found_in if found_in else 'NOT FOUND')

# the unanswerable question should not be covered by any document
print('\nDocs mentioning Lenovo or ThinkPad:',
      sum(('lenovo' in d['text'].lower() or 'thinkpad' in d['text'].lower()) for d in docs))

# one step from each expected answer in our Assignment 1 eval set
a1_answer_steps = [
    'Reinstall AnyConnect profile if mismatch',
    'Uncheck Account is locked out',
    'Raise capacity alert if disk > 90% after cleanup',
    'Restart network adapter in ncpa.cpl',
    'Inspect /var/log/secure for denied connection attempts',
    'Use Process Monitor to trace login process and identify bottlenecks',
    'Check for recent configuration changes or spam outbreaks',
    'Test on a clean workstation with minimal other software',
    'Restart print spooler service on workstation',
    'Ensure DFS (if used) is configured correctly and replicating',
]
found = sum(any(contains_span(d['text'], step) for d in docs) for step in a1_answer_steps)
print('Assignment 1 answer steps found in the corpus:', found, 'of', len(a1_answer_steps))

Q01 -> ['ubuntu_16_AppArmor - Ubuntu Server documentation.txt']
Q02 -> ['ubuntu_16_AppArmor - Ubuntu Server documentation.txt']
Q03 -> ['ubuntu_14_OpenSSH crypto configuration - Ubuntu Server docum.txt']
Q04 -> ['ubuntu_14_OpenSSH crypto configuration - Ubuntu Server docum.txt']
Q05 -> ['ubuntu_04_Install and configure PostgreSQL - Ubuntu Server d.txt']
Q06 -> ['ubuntu_04_Install and configure PostgreSQL - Ubuntu Server d.txt']
Q07 -> ['ubuntu_18_Docker storage networking and logging - Ubuntu Ser.txt']
Q08 -> ['ubuntu_18_Docker storage networking and logging - Ubuntu Ser.txt']
Q09 -> ['windows_autopilot_deployment.txt']
Q10 -> ['windows_autopilot_deployment.txt']

Docs mentioning Lenovo or ThinkPad: 0
Assignment 1 answer steps found in the corpus: 0 of 10


### Embedding model

`all-MiniLM-L6-v2` only takes 256 tokens, so our 400-token chunks would get cut off. We **switched the model** instead of changing the chunk size, and used `BAAI/bge-small-en-v1.5` for both sizes. It is a similar small model (384 dimensions) but takes up to 512 tokens. All token counts below use its tokenizer.

In [14]:
from sentence_transformers import SentenceTransformer

minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('all-MiniLM-L6-v2 max input tokens:', minilm.max_seq_length)
del minilm

EMBED_MODEL = 'BAAI/bge-small-en-v1.5'
embedder = SentenceTransformer(EMBED_MODEL, device=device)
tokenizer = embedder.tokenizer
print(EMBED_MODEL, 'max input tokens:', embedder.max_seq_length)
print('Vector size:', embedder.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (19 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BAAI/bge-small-en-v1.5 max input tokens: 512
Vector size: 384


/tmp/ipykernel_578/4033527820.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('Vector size:', embedder.get_sentence_embedding_dimension())


## A1. Build dense search

`make_chunks` splits each document into fixed-size token chunks with no overlap. We cut the original text using the tokenizer's character offsets, so chunks keep the normal text (decoding the tokens would lowercase everything).

`dense_search` is our reusable search function. It embeds the chunks and builds a FAISS `IndexFlatIP` the first time, then returns the top k chunk ids for a query. We normalise the vectors, so inner product = cosine similarity. bge recommends adding an instruction in front of the query (not the chunks).

In [15]:
import faiss

BATCH_SIZE = 64
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

def make_chunks(docs, chunk_size):
    chunks = []
    for d in docs:
        offsets = tokenizer(d['text'], add_special_tokens=False, return_offsets_mapping=True, verbose=False)['offset_mapping']
        for i in range(0, len(offsets), chunk_size):
            piece = offsets[i:i + chunk_size]
            start, end = piece[0][0], piece[-1][1]
            chunks.append({'doc_id': d['doc_id'], 'text': d['text'][start:end], 'n_tokens': len(piece)})
    return chunks

def dense_search(chunks, query, k=5, index=None):
    # build the index only once and pass it back in later calls
    if index is None:
        vectors = embedder.encode([c['text'] for c in chunks], batch_size=BATCH_SIZE,
                                  normalize_embeddings=True, show_progress_bar=False)
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors.astype('float32'))

    q_vec = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    scores, ids = index.search(q_vec.astype('float32'), k)
    return [int(i) for i in ids[0]], index

def is_hit(chunk_ids, chunks, span, k):
    return any(contains_span(chunks[i]['text'], span) for i in chunk_ids[:k])

## A2. Choose a chunk size

Dense search on the 10 queries with 100 and 400 token chunks. Hit rate@3 = % of queries where one of the top 3 chunks contains the answer span. Tokens sent per query = total tokens of the top 3 chunks, averaged over the queries.

In [16]:
chunks_by_size = {}
a2_rows = []

for size in [100, 400]:
    chunks = make_chunks(docs, size)
    index = None
    hits = 0
    tokens_sent = []
    for q in queries:
        top3, index = dense_search(chunks, q['query'], k=3, index=index)
        hit = is_hit(top3, chunks, q['span'], 3)
        hits += hit
        tokens_sent.append(sum(chunks[i]['n_tokens'] for i in top3))
        print(size, q['id'], 'hit' if hit else 'miss', [chunks[i]['doc_id'][:35] for i in top3])

    chunks_by_size[size] = chunks
    a2_rows.append({'Chunk size (tokens)': size,
                    'Total chunks': len(chunks),
                    'Hit rate@3 (%)': 100 * hits / len(queries),
                    'Tokens sent per query (top 3)': round(np.mean(tokens_sent))})

a2_table = pd.DataFrame(a2_rows)
a2_table

100 Q01 hit ['ubuntu_16_AppArmor - Ubuntu Server ', 'ubuntu_16_AppArmor - Ubuntu Server ', 'ubuntu_08_Glossary - Ubuntu Server ']
100 Q02 hit ['ubuntu_16_AppArmor - Ubuntu Server ', 'ubuntu_16_AppArmor - Ubuntu Server ', 'ubuntu_16_AppArmor - Ubuntu Server ']
100 Q03 miss ['ubuntu_14_OpenSSH crypto configurat', 'ubuntu_14_OpenSSH crypto configurat', 'ubuntu_14_OpenSSH crypto configurat']
100 Q04 miss ['ubuntu_14_OpenSSH crypto configurat', 'ubuntu_14_OpenSSH crypto configurat', 'ubuntu_14_OpenSSH crypto configurat']
100 Q05 hit ['ubuntu_04_Install and configure Pos', 'ubuntu_04_Install and configure Pos', 'ubuntu_04_Install and configure Pos']
100 Q06 miss ['ubuntu_04_Install and configure Pos', 'ubuntu_04_Install and configure Pos', 'ubuntu_04_Install and configure Pos']
100 Q07 hit ['ubuntu_18_Docker storage networking', 'ubuntu_18_Docker storage networking', 'ubuntu_18_Docker storage networking']
100 Q08 miss ['ubuntu_18_Docker storage networking', 'ubuntu_18_Docker storage networki

,Chunk size (tokens),Total chunks,Hit rate@3 (%),Tokens sent per query (top 3)
0,100,1658,50.0,296
1,400,432,90.0,1107


In [17]:
# keep the size with the better hit rate, the smaller one if they are equal
if a2_rows[1]['Hit rate@3 (%)'] > a2_rows[0]['Hit rate@3 (%)']:
    CHUNK_SIZE = 400
else:
    CHUNK_SIZE = 100
print('Chunk size used in Parts B-D:', CHUNK_SIZE)

Chunk size used in Parts B-D: 400


**A2.2 Answer:** We use 400-token chunks in Parts B to D. With this size dense search got a hit rate@3 of 90% compared to 50% for 100 tokens, and it sends about 1,107 tokens per query (top 3) instead of 296. Our Assignment 1 model has an 8192-token context window, but it was only trained on 512-token sequences, so 1,107 tokens of context is more than it has seen in training, and in practice it ignored the chunks (see D1). We therefore generate with a 7B instruct model and accept the higher token cost in return for much better retrieval.

### A1.2 Dense search details (for the final chunk set)

In [18]:
chunks = chunks_by_size[CHUNK_SIZE]

start = time.time()
_, dense_index = dense_search(chunks, queries[0]['query'])  # first call builds the index
build_time = time.time() - start

print('Embedding model   :', EMBED_MODEL)
print('Vector size       :', dense_index.d)
print('Similarity        : cosine (inner product on normalised vectors, IndexFlatIP)')
print('Batch size        :', BATCH_SIZE)
print('Chunks indexed    :', dense_index.ntotal)
print('Index build time  : %.2f s' % build_time)

Embedding model   : BAAI/bge-small-en-v1.5
Vector size       : 384
Similarity        : cosine (inner product on normalised vectors, IndexFlatIP)
Batch size        : 64
Chunks indexed    : 432
Index build time  : 3.57 s


# Part B - Keyword and hybrid search

Same chunks, same 10 queries, top 5 for each method.
- BM25 uses `rank_bm25` with simple lowercase word tokens.
- Hybrid uses Reciprocal Rank Fusion on the top 20 of dense and BM25: `RRF(d) = 1/(60 + dense rank) + 1/(60 + BM25 rank)`. Ranks start at 1, and a chunk that is missing from a list gets 0 from that list.

Latency is measured for returning the top 5, averaged over the 10 queries, after one warm-up query.

In [19]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

bm25 = BM25Okapi([tokenize(c['text']) for c in chunks])

def dense_top(query, k=5):
    ids, _ = dense_search(chunks, query, k, index=dense_index)
    return ids

def bm25_top(query, k=5):
    scores = bm25.get_scores(tokenize(query))
    return [int(i) for i in np.argsort(scores)[::-1][:k]]

def hybrid_top(query, k=5):
    rrf_scores = {}
    for ranked_list in [dense_top(query, 20), bm25_top(query, 20)]:
        for rank, chunk_id in enumerate(ranked_list, start=1):
            rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (60 + rank)
    ranked = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
    return ranked[:k]

methods = {'Dense': dense_top, 'BM25': bm25_top, 'Hybrid (RRF)': hybrid_top}

In [20]:
b_rows = []
top5 = {}

for name, search in methods.items():
    search(queries[0]['query'])  # warm-up
    times = []
    hit1 = hit3 = 0
    top5[name] = {}
    for q in queries:
        start = time.perf_counter()
        ids = search(q['query'], 5)
        times.append((time.perf_counter() - start) * 1000)
        top5[name][q['id']] = ids
        hit1 += is_hit(ids, chunks, q['span'], 1)
        hit3 += is_hit(ids, chunks, q['span'], 3)
    b_rows.append({'Method': name,
                   'Average query latency (ms)': round(np.mean(times), 2),
                   'Hit rate@1 (%)': 100 * hit1 / len(queries),
                   'Hit rate@3 (%)': 100 * hit3 / len(queries)})

b_table = pd.DataFrame(b_rows)
b_table

,Method,Average query latency (ms),Hit rate@1 (%),Hit rate@3 (%)
0,Dense,12.12,70.0,90.0
1,BM25,1.73,90.0,90.0
2,Hybrid (RRF),12.70,80.0,90.0


In [21]:
# rank of the first chunk that contains the span (0 = not in top 5)
def first_hit_rank(ids, span):
    for rank, i in enumerate(ids, start=1):
        if contains_span(chunks[i]['text'], span):
            return rank
    return 0

rank_rows = []
for q in queries:
    row = {'id': q['id']}
    for name in methods:
        row[name] = first_hit_rank(top5[name][q['id']], q['span'])
    rank_rows.append(row)
pd.DataFrame(rank_rows)

,id,Dense,BM25,Hybrid (RRF)
0,Q01,1,1,1
1,Q02,1,1,1
2,Q03,5,5,5
3,Q04,1,1,1
4,Q05,1,1,1
5,Q06,1,1,1
6,Q07,1,1,1
7,Q08,1,1,1
8,Q09,2,1,1
9,Q10,2,1,2


In [22]:
# best method = highest hit rate@3, then hit rate@1
best_method = max(b_rows, key=lambda r: (r['Hit rate@3 (%)'], r['Hit rate@1 (%)']))['Method']
print('Best method:', best_method)

# example query for B4 - pick one from the rank table above where the methods differ
EXAMPLE_ID = 'Q02'
example = [q for q in queries if q['id'] == EXAMPLE_ID][0]
print('\n' + example['query'])
for name in methods:
    print('\n' + name)
    for rank, i in enumerate(top5[name][EXAMPLE_ID][:3], start=1):
        mark = 'HIT ' if contains_span(chunks[i]['text'], example['span']) else '    '
        print(rank, mark, chunks[i]['doc_id'][:40], '|', chunks[i]['text'][:120])

Best method: BM25

How do you fully disable AppArmor on Ubuntu 24.04 LTS or later?

Dense
1 HIT  ubuntu_16_AppArmor - Ubuntu Server docum | u 24.04 LTS and later, the AppArmor services are baked into the Ubuntu Kernel. In earlier versions of Ubuntu, you could 
2      ubuntu_16_AppArmor - Ubuntu Server docum | Armor¶ Remove the apparmor=0 item from GRUB_CMDLINE_LINUX in /etc/default/grub. Run sudo update-grub to update your syst
3      ubuntu_16_AppArmor - Ubuntu Server docum | EMU/libvirt live migration QEMU machine types Container tools overview About rock images Docker storage, networking, and

BM25
1 HIT  ubuntu_16_AppArmor - Ubuntu Server docum | u 24.04 LTS and later, the AppArmor services are baked into the Ubuntu Kernel. In earlier versions of Ubuntu, you could 
2      ubuntu_06_Confidential Computing with AM | for addressing. As a result, the usable physical address space is reduced by one bit; this is commonly set to 1. kernel-
3      ubuntu_05_LXD containers and virtual mac |

**B4 Answer:** BM25 worked best on our queries (hit rate@1 90%, hit rate@3 90%), ahead of hybrid (80% / 90%) and dense (70% / 90%), and it was also about 7 times faster (1.7 ms against 12.1 ms for dense and 12.7 ms for hybrid). Our questions contain many exact technical terms such as command names, file paths and option names (for example `apparmor=0` and `GRUB_CMDLINE_LINUX` in Q02, "Autopilot" in Q09), which BM25 matches literally while dense embeddings blur them. For example, for Q09 the answer chunk was at rank 1 with BM25 and hybrid but at rank 2 with dense (see the rank table above). Hybrid sits in between because fusing with the weaker dense list pushed Q10 from rank 1 (BM25) to rank 2. With only 10 queries, each difference is one or two queries, and all three methods miss Q03 in the top 3 (see D2.3).

# Part C - Reranking

We take the best method from Part B, get the top 10 for each query, score them with the cross-encoder `cross-encoder/ms-marco-MiniLM-L-6-v2` and keep the top 3. Reranking time is only the cross-encoder scoring, after one warm-up.

In [23]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512, device=device)
best_search = methods[best_method]

def rerank(query, chunk_ids, keep=3):
    pairs = [(query, chunks[i]['text']) for i in chunk_ids]
    scores = reranker.predict(pairs)
    order = np.argsort(scores)[::-1]
    return [chunk_ids[j] for j in order[:keep]]

rerank(queries[0]['query'], best_search(queries[0]['query'], 10))  # warm-up

before = {}
after = {}
rerank_times = []
for q in queries:
    top10 = best_search(q['query'], 10)
    start = time.perf_counter()
    after[q['id']] = rerank(q['query'], top10)
    rerank_times.append((time.perf_counter() - start) * 1000)
    before[q['id']] = top10[:3]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [24]:
def show(chunk_id, span):
    mark = 'HIT' if contains_span(chunks[chunk_id]['text'], span) else '   '
    return mark + ' ' + chunks[chunk_id]['doc_id'][:40] + ' | ' + chunks[chunk_id]['text'][:80]

for q in queries:
    print('=' * 100)
    print(q['id'], q['query'])
    print('Before reranking:')
    for rank, i in enumerate(before[q['id']], start=1):
        print('  ', rank, show(i, q['span']))
    print('After reranking:')
    for rank, i in enumerate(after[q['id']], start=1):
        print('  ', rank, show(i, q['span']))

changed = sum(before[q['id']][0] != after[q['id']][0] for q in queries)
print('\nRank change rate: %d%%' % (100 * changed / len(queries)))

Q01 What are the two modes of operation of AppArmor profiles, and how do they differ?
Before reranking:
   1 HIT ubuntu_16_AppArmor - Ubuntu Server docum | EMU/libvirt live migration QEMU machine types Container tools overview About roc
   2     ubuntu_08_Glossary - Ubuntu Server docum | and application deployment. See also: The Ansible website Related topic(s): Auto
   3     ubuntu_18_Docker storage networking and  | answer for some of those problems: they are volumes, bind mounts and tmpfs. Anot
After reranking:
   1 HIT ubuntu_16_AppArmor - Ubuntu Server docum | EMU/libvirt live migration QEMU machine types Container tools overview About roc
   2     ubuntu_16_AppArmor - Ubuntu Server docum | the actual executable file path. For example, for the ping command use /bin/ping
   3     ubuntu_16_AppArmor - Ubuntu Server docum | (1571868402.378:24425): apparmor="DENIED" operation="open" profile="/usr/sbin/cu
Q02 How do you fully disable AppArmor on Ubuntu 24.04 LTS or later?
Before rerank

In [25]:
hit1_before = [r for r in b_rows if r['Method'] == best_method][0]['Hit rate@1 (%)']
hit1_after = 100 * sum(is_hit(after[q['id']], chunks, q['span'], 1) for q in queries) / len(queries)

c_table = pd.DataFrame({
    'Measure': ['Reranking time per query (ms)',
                'Rank change rate (%)',
                'Hit rate@1 before reranking (from Part B)',
                'Hit rate@1 after reranking'],
    'Value': [round(np.mean(rerank_times), 1),
              100 * changed / len(queries),
              hit1_before,
              hit1_after]})
c_table

,Measure,Value
0,Reranking time per query (ms),55.8
1,Rank change rate (%),10.0
2,Hit rate@1 before reranking (from Part B),90.0
3,Hit rate@1 after reranking,90.0


**C4 Answer:** Reranking did not help in our domain. Hit rate@1 stayed at 90% before and after reranking, and the first result changed for only 10% of the queries (Q03, where neither the old nor the new first result contained the answer); for the other queries it only reordered ranks 2 and 3. It added about 55.8 ms per query, roughly 32 times the BM25 search time (1.7 ms), which is small next to the time a 7B model needs to generate an answer but buys no accuracy here, so it is not worth keeping for our corpus.

# Part D - Generate and check answers

## D1. Grounded answers

**D1.1 Prompt.** The model has to answer only from the numbered chunks, cite the chunk numbers, and say "Not found in the documents." if the answer is not in them. For our Assignment 1 model we put it in the same `User: ... Assistant:` format we used for fine-tuning.

In [26]:
INSTRUCTIONS = ('You are an IT support assistant. Answer the question using ONLY the numbered chunks below. '
                'Do not use any outside knowledge. Cite the chunk number you used in square brackets, like [1] or [2]. '
                'If the chunks do not contain the answer, reply exactly: Not found in the documents.')

def build_context(chunk_ids):
    parts = []
    for n, i in enumerate(chunk_ids, start=1):
        parts.append('[' + str(n) + '] (source: ' + chunks[i]['doc_id'] + ')\n' + chunks[i]['text'])
    return '\n\n'.join(parts)

def user_message(query, chunk_ids):
    return 'Chunks:\n' + build_context(chunk_ids) + '\n\nQuestion: ' + query + '\n\nAnswer:'

def a1_prompt(query, chunk_ids):
    return 'User: ' + INSTRUCTIONS + '\n\n' + user_message(query, chunk_ids) + '\nAssistant:\n'

# top 3 after reranking for every query, plus the unanswerable one
context_ids = dict(after)
context_ids['U01'] = rerank(unanswerable['query'], best_search(unanswerable['query'], 10))
all_queries = queries + [unanswerable]

print(a1_prompt(queries[0]['query'], context_ids['Q01']))

User: You are an IT support assistant. Answer the question using ONLY the numbered chunks below. Do not use any outside knowledge. Cite the chunk number you used in square brackets, like [1] or [2]. If the chunks do not contain the answer, reply exactly: Not found in the documents.

Chunks:
[1] (source: ubuntu_16_AppArmor - Ubuntu Server documentation.txt)
EMU/libvirt live migration QEMU machine types Container tools overview About rock images Docker storage, networking, and logging About OpenStack Clouds About cloud-init Cloud images High Availability Introduction to HA Pacemaker resource agents Pacemaker fence agents Performance Profile-Guided Optimization hwloc CPU Power (states) cpupower (tool) TuneD CPU affinity and NUMA Contributing Contribute to this documentation Types of contributions Local development Guidance for writing AI tools in this project Getting help Our contributors Back to top Contribute to this page AppArmor¶ AppArmor is an easy-to-use Linux Security Module implem

In [27]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def generate(model, tok, prompt, max_new_tokens=256, repetition_penalty=1.0, add_special_tokens=True):
    inputs = tok(prompt, return_tensors='pt', add_special_tokens=add_special_tokens).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                repetition_penalty=repetition_penalty, pad_token_id=tok.eos_token_id)
    answer = tok.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    # our A1 model sometimes keeps going with a new 'User:' / 'Assistant:' turn
    answer = answer.split('\nUser:')[0].split('\nAssistant:')[0]
    return answer.strip()

def load_4bit(model_id):
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto')
    return model, tok

def chat_prompt(tok, system, user):
    messages = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

**D1.2 / D1.3 with our Assignment 1 model.** Same greedy decoding and repetition penalty (1.1) as in Assignment 1.

In [28]:
a1_tok = AutoTokenizer.from_pretrained(A1_MODEL_PATH)
a1_model = AutoModelForCausalLM.from_pretrained(A1_MODEL_PATH, torch_dtype=torch.float16, device_map='auto')
a1_model.eval()

a1_answers = {}
for q in all_queries:
    prompt = a1_prompt(q['query'], context_ids[q['id']])
    a1_answers[q['id']] = generate(a1_model, a1_tok, prompt, repetition_penalty=1.1)
    print('=' * 100)
    print(q['id'], q['query'])
    print('-' * 100)
    print(a1_answers[q['id']])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Q01 What are the two modes of operation of AppArmor profiles, and how do they differ?
----------------------------------------------------------------------------------------------------
Question: How does AppArmor work?

Answer:
Q02 How do you fully disable AppArmor on Ubuntu 24.04 LTS or later?
----------------------------------------------------------------------------------------------------
You are an IT support assistant. Answer the question using ONLY the numbered chunks below. Do not use any outside knowledge. Cite the chunk number you used in square brackets, like [1] or [2]. If the chunks do not contain the answer, reply exactly: Not found in the documents.

Chunks:
[1] (source: ubuntu_16_AppArmor - Ubuntu Server documentation.txt)
u 24.04 LTS and later, the AppArmor services are baked into the Ubuntu Kernel. In earlier versions of Ubuntu, you could disable AppArmor by not loading the service. However, it now requires setting a module parameter on the kernel command line to f

In [29]:
# does the A1 model follow the prompt?
says_not_found = 'not found in the documents' in a1_answers['U01'].lower()
cited = sum(bool(re.search(r'\[[123]\]', a1_answers[q['id']])) for q in queries)
print('Says Not found for U01:', says_not_found)
print('Answers that cite a chunk:', cited, 'of', len(queries))

# if it ignores the chunks or never says Not found we switch to an instruct model (allowed in the brief)
USE_INSTRUCT_MODEL = (not says_not_found) or cited < 5
print('Switch to instruct model:', USE_INSTRUCT_MODEL)

del a1_model
torch.cuda.empty_cache()

Says Not found for U01: False
Answers that cite a chunk: 4 of 10
Switch to instruct model: True


In [30]:
if USE_INSTRUCT_MODEL:
    GEN_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
    gen_model, gen_tok = load_4bit(GEN_MODEL)
    answers = {}
    for q in all_queries:
        prompt = chat_prompt(gen_tok, INSTRUCTIONS, user_message(q['query'], context_ids[q['id']]))
        answers[q['id']] = generate(gen_model, gen_tok, prompt, add_special_tokens=False)
    del gen_model
    torch.cuda.empty_cache()
else:
    GEN_MODEL = 'Assignment 1 model (final_it_qlora_merged)'
    answers = a1_answers

print('Generator used:', GEN_MODEL)
for q in all_queries:
    print('=' * 100)
    print(q['id'], q['query'])
    print('Chunks:', [chunks[i]['doc_id'][:40] for i in context_ids[q['id']]])
    print('-' * 100)
    print(answers[q['id']])

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Generator used: Qwen/Qwen2.5-7B-Instruct
Q01 What are the two modes of operation of AppArmor profiles, and how do they differ?
Chunks: ['ubuntu_16_AppArmor - Ubuntu Server docum', 'ubuntu_16_AppArmor - Ubuntu Server docum', 'ubuntu_16_AppArmor - Ubuntu Server docum']
----------------------------------------------------------------------------------------------------
The two modes of operation of AppArmor profiles are Complaining/Learning and Enforced/Confined. In Complaining/Learning mode, profile violations are permitted and logged, which is useful for testing and developing new profiles. In Enforced/Confined mode, profile policy is enforced in addition to logging the violation. [1]
Q02 How do you fully disable AppArmor on Ubuntu 24.04 LTS or later?
Chunks: ['ubuntu_16_AppArmor - Ubuntu Server docum', 'ubuntu_16_AppArmor - Ubuntu Server docum', 'ubuntu_16_AppArmor - Ubuntu Server docum']
--------------------------------------------------------------------------------------------------

**Generator used:** Qwen/Qwen2.5-7B-Instruct (4-bit). We switched because our Assignment 1 model (SmolLM2-360M, QLoRA on 10 examples) ignored the chunks: it cited a chunk in only 4 of 10 answers and did not reply "Not found" for the unanswerable question. The problem we saw: for U01 it invented a long, looping ThinkPad BIOS procedure ("Check if the password was correctly entered... repeat the process until the password is correct"), and for Q01 it wrote "Question: How does AppArmor work? Answer:" and gave no answer (see the Assignment 1 model outputs above).

**D1.3:** for U01 the final model (Qwen2.5-7B-Instruct) replied: "Not found in the documents."

## D2. LLM judge

The judge has to be a different model from the generator. If we generate with Qwen2.5-7B-Instruct we judge with Llama 3.1 8B Instruct, otherwise with Qwen. Llama is a gated model, so you need to accept its licence on Hugging Face and add a token as the Colab secret `HF_TOKEN`. Without access, `microsoft/Phi-3.5-mini-instruct` can be used instead.

The judge gets the question, the expected answer, the 3 chunks and the answer. It checks two things, each with defined labels, and has to reply in a fixed format so we can read it with a regex. We use greedy decoding, which is the same as temperature 0.

In [32]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

JUDGE_MODEL = 'meta-llama/Llama-3.1-8B-Instruct' if USE_INSTRUCT_MODEL else 'Qwen/Qwen2.5-7B-Instruct'
print('Generator:', GEN_MODEL)
print('Judge    :', JUDGE_MODEL)
judge_model, judge_tok = load_4bit(JUDGE_MODEL)

Generator: Qwen/Qwen2.5-7B-Instruct
Judge    : meta-llama/Llama-3.1-8B-Instruct


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [33]:
JUDGE_PROMPT = '''You are checking an answer from an IT support assistant that must only use the given chunks.
You get the QUESTION, the EXPECTED ANSWER, the 3 CHUNKS the assistant saw, and the ASSISTANT ANSWER.

Check 1 - CORRECT (compare with the expected answer):
- CORRECT: has all the key facts of the expected answer and nothing that contradicts it. Wording can differ.
- PARTLY_CORRECT: has at least one key fact but misses another one, or mixes correct facts with a wrong detail.
- WRONG: has none of the key facts, contradicts the expected answer, or says 'Not found in the documents'.

Check 2 - SUPPORTED (compare with the 3 chunks only, not with the expected answer or your own knowledge):
- YES: every statement in the answer can be found in the chunks. An answer that only says 'Not found in the documents' counts as YES.
- NO: at least one statement is not in the chunks.

Reply in exactly this format:
CORRECT: <CORRECT, PARTLY_CORRECT or WRONG>
SUPPORTED: <YES or NO>
REASON: <one sentence>
UNSUPPORTED: <quote the unsupported statement, or NONE>'''

def judge_input(q, answer):
    return ('QUESTION:\n' + q['query'] +
            '\n\nEXPECTED ANSWER:\n' + q['answer'] +
            '\n\nCHUNKS:\n' + build_context(context_ids[q['id']]) +
            '\n\nASSISTANT ANSWER:\n' + answer)

def read_field(output, field):
    match = re.search(r'(?m)^\W*' + field + r'\W*:\s*(.*)$', output)
    return match.group(1).strip() if match else ''

print(JUDGE_PROMPT)

You are checking an answer from an IT support assistant that must only use the given chunks.
You get the QUESTION, the EXPECTED ANSWER, the 3 CHUNKS the assistant saw, and the ASSISTANT ANSWER.

Check 1 - CORRECT (compare with the expected answer):
- CORRECT: has all the key facts of the expected answer and nothing that contradicts it. Wording can differ.
- PARTLY_CORRECT: has at least one key fact but misses another one, or mixes correct facts with a wrong detail.
- WRONG: has none of the key facts, contradicts the expected answer, or says 'Not found in the documents'.

Check 2 - SUPPORTED (compare with the 3 chunks only, not with the expected answer or your own knowledge):
- YES: every statement in the answer can be found in the chunks. An answer that only says 'Not found in the documents' counts as YES.
- NO: at least one statement is not in the chunks.

Reply in exactly this format:
CORRECT: <CORRECT, PARTLY_CORRECT or WRONG>
SUPPORTED: <YES or NO>
REASON: <one sentence>
UNSUPPORTE

In [34]:
pd.set_option('display.max_colwidth', None)  # show the full reason text
judge_rows = []
for q in queries:
    prompt = chat_prompt(judge_tok, JUDGE_PROMPT, judge_input(q, answers[q['id']]))
    output = generate(judge_model, judge_tok, prompt, max_new_tokens=150, add_special_tokens=False)
    judge_rows.append({'id': q['id'],
                       'query': q['query'],
                       'correct': read_field(output, 'CORRECT').upper().replace(' ', '_'),
                       'supported': read_field(output, 'SUPPORTED').upper(),
                       'reason': read_field(output, 'REASON'),
                       'unsupported': read_field(output, 'UNSUPPORTED')})

judge_table = pd.DataFrame(judge_rows)
judge_table

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


,id,query,correct,supported,reason,unsupported
0,Q01,What are the two modes of operation of AppArmo...,CORRECT,YES,The assistant's answer is supported by chunk [...,NONE
1,Q02,How do you fully disable AppArmor on Ubuntu 24...,CORRECT,YES,The assistant's answer is supported by chunk [...,NONE
2,Q03,In OpenSSH algorithm lists such as Ciphers or ...,WRONG,YES,The assistant's answer is not supported by the...,NONE
3,Q04,How can I check the effective Ciphers setting ...,CORRECT,YES,The assistant's answer is supported by chunk [...,NONE
4,Q05,What are the default PostgreSQL client authent...,CORRECT,YES,The assistant's answer is supported by chunk [...,NONE
5,Q06,How do I allow other computers to connect to m...,CORRECT,YES,The assistant's answer is supported by chunk [...,NONE
6,Q07,What is the recommended way to persist data fr...,CORRECT,YES,The assistant's answer is supported by the pro...,NONE
7,Q08,Which logging driver does Docker use by defaul...,CORRECT,YES,The assistant's answer is supported by chunk [...,NONE
8,Q09,Does Windows Autopilot need to re-image new de...,CORRECT,YES,The assistant's answer is supported by chunk [...,NONE
9,Q10,What subscription is needed to configure Windo...,PARTLY_CORRECT,YES,The assistant's answer is partially correct as...,NONE


In [35]:
summary = pd.DataFrame({
    'Check': ['Correct', 'Partly correct', 'Wrong', 'Supported by the chunks'],
    'Count out of 10': [(judge_table['correct'] == 'CORRECT').sum(),
                        (judge_table['correct'] == 'PARTLY_CORRECT').sum(),
                        (judge_table['correct'] == 'WRONG').sum(),
                        (judge_table['supported'] == 'YES').sum()]})
summary

,Check,Count out of 10
0,Correct,8
1,Partly correct,1
2,Wrong,1
3,Supported by the chunks,10


### D2.2 Spot-check the judge
We read 3 answers and their chunks ourselves and compare with the judge.

In [36]:
for qid in ['Q02', 'Q05', 'Q08']:
    q = [x for x in queries if x['id'] == qid][0]
    row = [r for r in judge_rows if r['id'] == qid][0]
    print('#' * 100)
    print(qid, q['query'])
    print('Expected:', q['answer'])
    print('-' * 100)
    print(build_context(context_ids[qid]))
    print('-' * 100)
    print('Answer:', answers[qid])
    print('Judge :', row['correct'], '/', row['supported'], '-', row['reason'])

####################################################################################################
Q02 How do you fully disable AppArmor on Ubuntu 24.04 LTS or later?
Expected: From 24.04 AppArmor is built into the kernel, so you add apparmor=0 to GRUB_CMDLINE_LINUX in /etc/default/grub, run sudo update-grub and reboot.
----------------------------------------------------------------------------------------------------
[1] (source: ubuntu_16_AppArmor - Ubuntu Server documentation.txt)
u 24.04 LTS and later, the AppArmor services are baked into the Ubuntu Kernel. In earlier versions of Ubuntu, you could disable AppArmor by not loading the service. However, it now requires setting a module parameter on the kernel command line to fully disable or re-enable AppArmor. Disable AppArmor¶ Warning Disabling AppArmor reduces the security of your system! You should only disable AppArmor if you understand the security implications of disabling the service! To disable AppArmor, you must do the fo

| Query | Judge (correct / supported) | Our verdict | Agree? |
| --- | --- | --- | --- |
| Q02 | CORRECT / YES | correct / supported | Yes |
| Q05 | CORRECT / YES | correct / supported | Yes |
| Q08 | CORRECT / YES | correct / supported | Yes |

**D2.2 Answer:** We agree with all 3 of the 3 verdicts: in each case the answer gives the facts of the expected answer (the apparmor=0 steps for Q02, peer and scram-sha-256 for Q05, json-file writing JSON logs to a file for Q08) and each of those facts is in the answer's chunks. We found no disagreement in these 3. In the case we look at in D2.3 (Q03), the judge's labels (WRONG / YES) are right, because "Not found" for an answerable question is wrong and a "Not found" reply counts as supported, but its one-sentence reason contradicts itself ("not supported by the chunks, but it is correct in the sense that it is not a wrong answer"), so the labels are more reliable than the reasons from this 8B judge.

### D2.3 Find the cause
We take the first answer the judge marked WRONG or not supported, and check whether the answer span was in its 3 chunks. If it was, the problem is in generation. If not, retrieval did not find it.

In [37]:
failed = [r for r in judge_rows if r['correct'] == 'WRONG' or r['supported'] == 'NO']
print('Failed answers:', [r['id'] for r in failed])

if failed:
    fail_id = failed[0]['id']
    q = [x for x in queries if x['id'] == fail_id][0]
    print('\n' + fail_id, q['query'])
    print('Expected:', q['answer'])
    print('Answer  :', answers[fail_id])
    print('Judge   :', failed[0]['correct'], '/', failed[0]['supported'], '-', failed[0]['reason'])
    print('-' * 100)
    print(build_context(context_ids[fail_id]))
    print('-' * 100)
    print('Answer span in the 3 chunks:', is_hit(context_ids[fail_id], chunks, q['span'], 3))

    # where did the right chunk end up in the search results?
    for name, search in methods.items():
        print(name, 'rank in top 20:', first_hit_rank(search(q['query'], 20), q['span']))

Failed answers: ['Q03']

Q03 In OpenSSH algorithm lists such as Ciphers or MACs, what do the +, - and ^ prefixes do?
Expected: + appends the algorithms to the end of the default set, - removes them from the default set and ^ puts them at the beginning of the default set. A list without a prefix replaces the default set.
Answer  : Not found in the documents.
Judge   : WRONG / YES - The assistant's answer is not supported by the chunks, but it is correct in the sense that it is not a wrong answer.
----------------------------------------------------------------------------------------------------
[1] (source: ubuntu_14_OpenSSH crypto configuration - Ubuntu Server docum.txt)
the server and the client. Ciphers List of symmetric ciphers. Examples include aes256-ctr and chacha20-poly1305@openssh.com. MACs List of Message Authentication Code algorithms, used for data integrity protection. The -etm versions calculate the MAC after encryption and are considered safer. Examples include hmac-sha2

**D2.3 Answer:** For Q03 the answer span was not in the 3 chunks, so the problem was caused by retrieval, not generation. The answer is in the OpenSSH page ("+ The specified algorithm(s) will be appended to the end of the default set"), but dense search, BM25 and hybrid all ranked the chunk with it only 5th, and the reranker did not move it into the top 3. Generation behaved correctly: with chunks that lack the information, the model replied "Not found in the documents". The query is made of symbols (+, - and ^), which BM25's tokenizer deletes and embeddings represent poorly, which explains the weak ranking.

### D2.4 Suggest a fix

**Answer:** Give the generator the top 5 chunks instead of the top 3. The chunk with the answer is already within the top 5 for dense, BM25 and hybrid search, so passing 5 chunks (about 2,000 tokens, which Qwen2.5-7B-Instruct handles easily) would put it in front of the model and let it answer Q03, while the same instruction "answer only from the chunks" keeps the model grounded.

## Export
Run all cells so every output is visible, then use **File > Download > Download .ipynb**. Make the HTML copy on your computer with `jupyter nbconvert --to html Assignment2B_RAG_Pipeline.ipynb`, and check that the outputs appear in it.


In [38]:
# HTML export is done on your own computer after downloading the .ipynb (see the cell above)
